# Nettoyage initial et exploration catégorielle - ASFIM

Ce notebook charge `asfim_performances_consolidees.csv`, supprime les colonnes de performance courte demandées, vérifie les sociétés de gestion contre les sources officielles ASFIM / AMMC, puis fait une exploration statistique des variables catégorielles.

Sources de référence utilisées pour le contrôle des noms :
- ASFIM, page `Nos membres`
- AMMC, fiche société pour `ADVISORY AND FINANCE ASSET MANAGEMENT`

Date de référence du contrôle : 2026-06-08.

In [ ]:
from pathlib import Path
import re
import unicodedata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 140)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120

In [ ]:
csv_path = Path("asfim_performances_consolidees.csv")
df_raw = pd.read_csv(csv_path, encoding="utf-8-sig", low_memory=False, parse_dates=["date"])

print(f"Shape brute : {df_raw.shape}")
print(f"Période couverte : {df_raw['date'].min().date()} -> {df_raw['date'].max().date()}")
print(f"Sociétés de gestion uniques (brut) : {df_raw['societe_gestion'].nunique()}")
print("Colonnes :")
print(df_raw.columns.tolist())

display(df_raw.head())

In [ ]:
perf_cols = [
    "perf_1j", "perf_1s", "perf_1m", "perf_3m", "perf_6m",
    "perf_1a", "perf_2a", "perf_3a", "perf_5a",
]

df = df_raw.drop(columns=perf_cols).copy()
assert not set(perf_cols).intersection(df.columns)

print(f"Shape après suppression des colonnes de performance : {df.shape}")
print("Colonnes conservées :")
print(df.columns.tolist())

display(df.head())

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0].to_frame(name="missing_count")
missing["missing_pct"] = (missing["missing_count"] / len(df) * 100).round(2)

print("Colonnes avec valeurs manquantes :")
display(missing)

In [ ]:
def normalize_label(value):
    if pd.isna(value):
        return ""
    text = str(value).strip().upper()
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("ascii")
    text = re.sub(r"\s+", " ", text)
    return text

# Liste actuelle des membres ASFIM visible sur la page officielle "Nos membres".
current_asfim_members = [
    "AD Capital ASSET MANAGEMENT",
    "AFG Asset Management",
    "Africapital Management",
    "Alphavest Asset Management",
    "Atlas Capital Management",
    "BMCE Capital Gestion",
    "CIH CAPITAL MANAGEMENT",
    "Capital Trust Gestion",
    "CDG Capital Gestion",
    "CFG Gestion",
    "IRG Asset Management",
    "Marogest",
    "QUANTUM CAPITAL GESTION",
    "Red Med Asset Management",
    "RMA Asset Management",
    "SAHAM CAPITAL GESTION",
    "STERLING ASSET MANAGEMENT",
    "Twin Capital Gestion",
    "Upline Capital Management",
    "Valoris Management",
    "Wafa Gestion",
    "Wineo Gestion",
]

csv_societies = sorted(df["societe_gestion"].dropna().unique(), key=normalize_label)
csv_norm = {normalize_label(name): name for name in csv_societies}
current_norm = {normalize_label(name): name for name in current_asfim_members}

absent_from_asfim = sorted(set(csv_norm) - set(current_norm))
missing_from_csv = sorted(set(current_norm) - set(csv_norm))

print("Sociétés présentes dans le CSV mais absentes de la liste ASFIM actuelle :")
print([csv_norm[n] for n in absent_from_asfim])
print()
print("Sociétés de la liste ASFIM actuelle absentes du CSV :")
print([current_norm[n] for n in missing_from_csv])

# La seule différence de dénomination notable trouvée dans les sources officielles est un nom légal plus complet sur l'AMMC.
official_name_difference = pd.DataFrame([
    {
        "nom_dans_le_csv": "AFG ASSET MANAGEMENT",
        "denomination_officielle_AMMC": "ADVISORY AND FINANCE ASSET MANAGEMENT",
        "commentaire": "Même entité; le CSV utilise la forme courte / commerciale, la fiche AMMC donne la dénomination légale complète.",
    }
])

display(official_name_difference)

In [ ]:
categorical_cols = [
    "societe_gestion",
    "nature_juridique",
    "classification",
    "sensibilite",
    "indice_benchmark",
    "periodicite_vl",
    "souscripteurs",
    "affectation_resultats",
    "depositaire",
    "reseau_placeur",
]


def categorical_overview(data, cols):
    rows = []
    for col in cols:
        s = data[col]
        mode = s.mode(dropna=True)
        top_value = mode.iat[0] if not mode.empty else np.nan
        counts = s.value_counts(dropna=True)
        top_freq = int(counts.iat[0]) if not counts.empty else 0
        rows.append(
            {
                "colonne": col,
                "dtype": str(s.dtype),
                "non_null": int(s.notna().sum()),
                "missing": int(s.isna().sum()),
                "missing_%": round(s.isna().mean() * 100, 2),
                "n_unique": int(s.nunique(dropna=True)),
                "mode": top_value,
                "mode_freq": top_freq,
                "mode_%": round(top_freq / len(data) * 100, 2),
            }
        )
    return pd.DataFrame(rows).sort_values(["n_unique", "missing"], ascending=[False, False])

summary_cat = categorical_overview(df, categorical_cols)
display(summary_cat)

In [ ]:
def frequency_table(data, col, top=None):
    s = data[col].fillna("Manquant")
    counts = s.value_counts(dropna=False)
    if top is not None:
        counts = counts.head(top)
    out = counts.to_frame(name="effectif")
    out["part_%"] = (out["effectif"] / len(data) * 100).round(2)
    return out

frequency_specs = [
    ("societe_gestion", None),
    ("nature_juridique", None),
    ("classification", None),
    ("periodicite_vl", None),
    ("souscripteurs", None),
    ("affectation_resultats", None),
    ("depositaire", None),
    ("reseau_placeur", 10),
    ("sensibilite", 10),
    ("indice_benchmark", 15),
]

for col, top in frequency_specs:
    print(f"\n### {col}")
    display(frequency_table(df, col, top=top))

In [ ]:
# Quelques tableaux croisés utiles pour l'exploration
print("Nature juridique x classification")
display(pd.crosstab(df["nature_juridique"], df["classification"]))

print("\nNature juridique x classification (en % par ligne)")
display(pd.crosstab(df["nature_juridique"], df["classification"], normalize="index").round(3))

print("\nSociété de gestion x classification")
display(pd.crosstab(df["societe_gestion"], df["classification"]))

In [ ]:
def plot_bar(data, col, top=None, ax=None, title=None):
    counts = data[col].fillna("Manquant").value_counts(dropna=False)
    if top is not None:
        counts = counts.head(top)
    counts = counts.sort_values(ascending=True)
    sns.barplot(x=counts.values, y=counts.index, ax=ax, color="#2a9d8f")
    ax.set_title(title or col)
    ax.set_xlabel("Effectif")
    ax.set_ylabel("")

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
plot_bar(df, "nature_juridique", ax=axes[0, 0], title="Nature juridique")
plot_bar(df, "classification", ax=axes[0, 1], title="Classification")
plot_bar(df, "periodicite_vl", ax=axes[1, 0], title="Périodicité VL")
plot_bar(df, "souscripteurs", ax=axes[1, 1], title="Souscripteurs")
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
plot_bar(df, "societe_gestion", top=10, ax=ax, title="Top 10 des sociétés de gestion")
plt.tight_layout()

## Lecture rapide

- Aucune société de gestion de la liste actuelle ASFIM n'apparaît comme manquante dans le CSV après normalisation des majuscules / accents.
- Le seul écart de dénomination officiellement visible est `AFG ASSET MANAGEMENT` dans le CSV, qui correspond à la dénomination légale `ADVISORY AND FINANCE ASSET MANAGEMENT` sur l'AMMC.
- Les variables les plus structurantes sont `nature_juridique`, `classification`, `periodicite_vl`, `souscripteurs` et `affectation_resultats`.
- `sensibilite`, `indice_benchmark` et `reseau_placeur` montrent des libellés très hétérogènes et mériteront une normalisation plus poussée si on veut une analyse fine.